# Fine-tune the 13-band Sentinel-2 ResNet-18

This notebook loads `resnet18_sentinel2_all_moco.pth`, adds a 10-class EuroSAT head, and fine-tunes it on the repository's 13-band GeoTIFF dataset.

Expected layout: `data/raw/EuroSATallBands/<class_name>/*.tif` and `data/processed/splits.json`.

> On this Windows AMD laptop, PyTorch will normally use the CPU. CUDA is NVIDIA-only; the notebook therefore uses a smaller batch size and fewer epochs by default.

In [ ]:
# ==========================================
# CELL 1: Setup, Clone Repo & HuggingFace Setup
# ==========================================
!pip install -q huggingface_hub torchgeo scikit-learn matplotlib timm

import os
import json
import random
from pathlib import Path
from contextlib import nullcontext
from google.colab import files
from huggingface_hub import hf_hub_download, login

# Optional: Authenticate if your HF repository is private
# login(token="YOUR_HF_TOKEN")

# Clone your project repo to get custom modules (like src/train_spectral_resnet_torchgeo.py)
!git clone https://github.com/Airam18/Land-cover-classification-ROSPIN-Summer-School.git
%cd Land-cover-classification-ROSPIN-Summer-School

import importlib.util
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from torch.utils.data import DataLoader

# Device Setup (Google Colab T4 GPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Import custom training script from repository
REPO_ROOT = Path.cwd().resolve()
SCRIPT_PATH = REPO_ROOT / "src" / "train_spectral_resnet_torchgeo.py"
spec = importlib.util.spec_from_file_location("spectral_training", SCRIPT_PATH)
spectral = importlib.util.module_from_spec(spec)
spec.loader.exec_module(spectral)

# Download pretrained model weights from your Hugging Face repo
print("Downloading base model weights from Hugging Face...")
WEIGHTS_PATH = hf_hub_download(
    repo_id="Airam18/land-cover-clasification-model-all-bands",
    filename="resnet18_sentinel2_all_moco.pth"  # Replace with exact filename if different
)

CHECKPOINT_PATH = REPO_ROOT / "checkpoints" / "spectral_resnet_notebook_best.pth"
EPOCHS = 10
BATCH_SIZE = 32 if DEVICE.type == "cuda" else 8
PATIENCE = 3
LEARNING_RATE = 1e-4

print("Loaded Base Weights Path:", WEIGHTS_PATH)

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import random
from contextlib import nullcontext

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from torch.utils.data import DataLoader

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

SCRIPT_PATH = REPO_ROOT / "src" / "train_spectral_resnet_torchgeo.py"
spec = importlib.util.spec_from_file_location("spectral_training", SCRIPT_PATH)
spectral = importlib.util.module_from_spec(spec)
spec.loader.exec_module(spectral)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WEIGHTS_PATH = REPO_ROOT / "notebooks" / "resnet18_sentinel2_all_moco.pth"
CHECKPOINT_PATH = REPO_ROOT / "checkpoints" / "spectral_resnet_notebook_best.pth"
EPOCHS = 10
BATCH_SIZE = 8 if DEVICE.type == "cpu" else 32
PATIENCE = 3
LEARNING_RATE = 1e-4

if DEVICE.type == "cpu":
    torch.set_num_threads(min(8, os.cpu_count() or 4))
    print("CUDA is unavailable; using CPU-friendly settings.")

print("Device:", DEVICE)
print("Weights:", WEIGHTS_PATH)

CUDA is unavailable; using CPU-friendly settings.
Device: cpu
Weights: C:\Users\Mary\PycharmProjects\Land-cover-classification-ROSPIN-Summer-School\notebooks\resnet18_sentinel2_all_moco.pth


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
samples = spectral.find_samples()
with open(spectral.SPLITS_PATH, encoding="utf-8") as file:
    splits = json.load(file)

train_set = spectral.EuroSATMultispectral(samples, splits["train"], augment=True)
val_set = spectral.EuroSATMultispectral(samples, splits["val"], augment=False)
test_set = spectral.EuroSATMultispectral(samples, splits["test"], augment=False)

loader_args = {
    "batch_size": BATCH_SIZE,
    "num_workers": 0,  # Keep 0 on Windows for reliability
    "pin_memory": DEVICE.type == "cuda",
}
train_loader = DataLoader(train_set, shuffle=True, **loader_args)
val_loader = DataLoader(val_set, shuffle=False, **loader_args)
test_loader = DataLoader(test_set, shuffle=False, **loader_args)
print(f"Samples: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}")

Samples: train=18900, val=4050, test=4050


In [ ]:
model = spectral.build_model(
    num_classes=len(spectral.CLASS_NAMES),
    pretrained=True,
    weights_path=str(WEIGHTS_PATH),
).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
print(model)

Local weights loaded from C:\Users\Mary\PycharmProjects\Land-cover-classification-ROSPIN-Summer-School\notebooks\resnet18_sentinel2_all_moco.pth
Missing keys: ['fc.weight', 'fc.bias']
Unexpected keys: []
ResNet(
  (conv1): Conv2d(13, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (drop_block): Identity()
      (act1): ReLU(inplace=True)
      (aa): Identity()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine

In [ ]:
def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    correct = 0
    total = 0
    autocast = (torch.autocast(device_type="cuda", dtype=torch.float16)
                if DEVICE.type == "cuda" else nullcontext())

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with autocast:
            logits = model(images)
            loss = criterion(logits, labels)
        if training:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

history = {"train_loss": [], "train_accuracy": [], "val_loss": [], "val_accuracy": []}
best_val_accuracy = -1.0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    val_loss, val_accuracy = run_epoch(val_loader, training=False)
    scheduler.step()
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)
    print(f"Epoch {epoch:02d}/{EPOCHS} | train acc={train_accuracy:.4f} | val acc={val_accuracy:.4f}")

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        epochs_without_improvement = 0
        CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            "model_state_dict": model.state_dict(),
            "classes": spectral.CLASS_NAMES,
            "bands": spectral.BAND_ORDER,
            "input_size": spectral.INPUT_SIZE,
            "epoch": epoch,
            "val_accuracy": val_accuracy,
        }, CHECKPOINT_PATH)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping")
            break

KeyboardInterrupt: 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].legend()
axes[1].plot(history["train_accuracy"], label="train")
axes[1].plot(history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy")
axes[1].legend()
plt.show()

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        predictions = model(images.to(DEVICE)).argmax(dim=1).cpu().numpy()
        y_pred.extend(predictions)
        y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred, target_names=spectral.CLASS_NAMES))
matrix = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(matrix, display_labels=spectral.CLASS_NAMES).plot(
    xticks_rotation=45, cmap="Blues", values_format="d"
)
plt.tight_layout()
plt.show()
print("Best checkpoint:", CHECKPOINT_PATH)